# Cardiology Multilabel Classification

This notebook shows how to build a multilabel ECG classification pipeline with the PhysioNet 2020 cardiology dataset in PyHealth. It loads the dataset, applies the `CardiologyMultilabelClassification` task, inspects example samples, and includes a small Member 2 sanity check that verifies a model can run a forward pass and backpropagate on a tiny 5-sample subset.

Download link: https://physionet.org/content/challenge-2020/1.0.2/
You'll need to run the following in terminal:

`cd ~/data && wget -r -N -c -np https://physionet.org/files/challenge-2020/1.0.2/`

## 1. Install Deps

In [ ]:
%pip install -q -e ..
%load_ext autoreload
%autoreload 2

## 2. Load the Dataset

In [ ]:
from pathlib import Path

DATA_ROOT = str(Path.home() / "data" / "physionet.org" / "files" / "challenge-2020" / "1.0.2" / "training")
print(f"DATA_ROOT = {DATA_ROOT}")

In [ ]:
# Optional: clear the cache to force a full rebuild
# If you decide to change any of the core code this will be necessary to pick up changes
import shutil, os
cache_dir = "/tmp/pyhealth_cardiology"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"Cleared cache: {cache_dir}")
else:
    print(f"No cache found at {cache_dir}")

In [ ]:
from pyhealth.datasets import Cardiology2Dataset

dataset = Cardiology2Dataset(
    root=DATA_ROOT,
    chosen_dataset=[1, 1, 0, 0, 0, 0], # Only load cpsc_2018 datasets
    dev=True,
    cache_dir="/tmp/pyhealth_cardiology"
)

dataset.stats()

## 3. Apply the Multilabel Classification Task

In [ ]:
from pyhealth.tasks import CardiologyMultilabelClassification

task = CardiologyMultilabelClassification()
sample_dataset = dataset.set_task(task)

print(f"Total samples: {len(sample_dataset)}")

## 4. Inspect Sample

In [ ]:
sample = sample_dataset[0]
print(f"keys:       {list(sample.keys())}")
print(f"patient_id: {sample['patient_id']}")
print(f"visit_id:   {sample['visit_id']}")
print(f"signal:     {sample['signal']}")
print(f"labels:     {sample['labels']}")

## 5. Member 2 Sanity Check

In [ ]:
from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import SparcNet

mini_samples = [sample_dataset[i] for i in range(min(5, len(sample_dataset)))]
mini_dataset = create_sample_dataset(
    samples=mini_samples,
    input_schema=task.input_schema,
    output_schema=task.output_schema,
    dataset_name="cardiology_multilabel_mini",
)

model = SparcNet(
    dataset=mini_dataset,
    block_layers=2,
    growth_rate=8,
    bn_size=4,
    drop_rate=0.1,
)

batch = next(iter(get_dataloader(mini_dataset, batch_size=len(mini_samples), shuffle=False)))
ret = model(**batch)

print({k: tuple(v.shape) if hasattr(v, "shape") else type(v) for k, v in ret.items()})
ret["loss"].backward()

has_gradient = any(
    param.requires_grad and param.grad is not None for param in model.parameters()
)
print(f"Gradient check passed: {has_gradient}")